
# Document Ingestion

## Objective
To understand how documents are loaded, validated, and converted into text that can be processed by a Generative AI application.

## Experiments Covered
- Extract text from PDF documents.
- Identify scanned PDFs that do not contain selectable text.
- Apply Optical Character Recognition (OCR).
- Clean extracted text.
- Extract metadata from documents.
- Validate document files.
- Extract text from PDF, DOCX, and TXT formats.
- Handle invalid, missing, and empty documents.
- Build a unified, metadata-aware document ingestion pipeline.

## Tools and Libraries
- Python
- PyMuPDF
- Tesseract OCR
- pytesseract
- Pillow
- python-docx
- Regular expressions



## 1. Install the Required Python Libraries

The following libraries are used to extract text from different document formats and process scanned documents.

- PyMuPDF: Reads PDF files and extracts text.
- Pillow: Handles images generated from PDF pages.
- pytesseract: Connects Python to the Tesseract OCR engine.
- python-docx: Reads Microsoft Word documents.
- reportlab: Creates sample PDF files for experimentation.

Note: Tesseract OCR must also be installed separately on the computer.


In [83]:
%pip install pymupdf pillow reportlab pytesseract python-docx

Note: you may need to restart the kernel to use updated packages.



## 2. Import Libraries and Configure OCR

Import the libraries required for PDF extraction, image processing, OCR, Word document extraction, text cleaning, and file validation.

The Tesseract executable path must match the installation location on the computer.


In [87]:
import os
import re
import pymupdf
import pytesseract

from PIL import Image
from docx import Document

# Configure the Tesseract executable
pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Users\vishakha.more\AppData\Local"
    r"\Tesseract-OCR\tesseract.exe"
)

print("PyMuPDF version:", pymupdf.__version__)
print("Tesseract exists:", os.path.exists(
    pytesseract.pytesseract.tesseract_cmd
))

PyMuPDF version: 1.28.2
Tesseract exists: True



## 3. Prepare the Sample Documents

Create a folder named `Documents` to store sample files used in the experiments.

The experiments use:
- `sample.pdf`: A text-based PDF.
- `scanned_sample.pdf`: An image-based PDF.
- `sample.docx`: A Word document.
- `sample.txt`: A plain text file.
- `blank.pdf`: A PDF containing no extractable text.
- `invalid.pdf`: A file containing ordinary text but saved with a PDF extension.


In [88]:
os.makedirs("Documents", exist_ok=True)

print("Documents folder is ready.")
print("Existing files:", os.listdir("Documents"))

Documents folder is ready.
Existing files: ['.ipynb_checkpoints', 'blank.pdf', 'invalid.pdf', 'sample.docx', 'sample.pdf', 'sample.txt', 'scanned_page.png', 'scanned_sample.pdf']



## 4. Extract Text from a Normal PDF

A text-based PDF contains text that can usually be selected and copied.

PyMuPDF opens the PDF and extracts the text from each page using `page.get_text()`.


In [89]:
pdf_path = "Documents/sample.pdf"

with pymupdf.open(pdf_path) as document:
    for page_number, page in enumerate(document, start=1):
        print(f"\nPage {page_number}")
        print(page.get_text())


Page 1
Company Leave Policy - Sample PDF
Document ID: HR-POL-001
Version: 1.0
Annual Leave
Employees receive 20 days of annual leave per calendar year. Leave requests must be submitted
to the employee's manager for approval.
Sick Leave
Employees may take sick leave when they are unwell. They should notify their manager as soon as
reasonably possible.
Public Holidays
Public holidays are handled according to the company's applicable holiday calendar.
This is a fictional sample document created for practicing PDF text extraction. It is not a real company policy.




## 5. Check Whether a PDF Contains Extractable Text

A scanned PDF may contain images of text rather than actual text characters.

If `page.get_text()` returns an empty string, OCR may be required to recognize the text within the page image.


In [90]:
with pymupdf.open("Documents/scanned_sample.pdf") as document:
    text = document[0].get_text()

    print("Extracted text:", repr(text))

    if text.strip():
        print("Selectable text is available.")
    else:
        print("No selectable text found. OCR may be required.")

Extracted text: ''
No selectable text found. OCR may be required.



## 6. Extract Text from a Scanned PDF Using OCR

Optical Character Recognition (OCR) identifies text within an image.

The process is:
1. Open the scanned PDF.
2. Convert the PDF page into an image.
3. Pass the image to Tesseract OCR.
4. Display the recognized text.

OCR can introduce spelling or formatting errors, especially when the scanned image has low quality.


In [91]:
pdf_path = "Documents/scanned_sample.pdf"

with pymupdf.open(pdf_path) as document:
    page = document[0]

    pix = page.get_pixmap(
        dpi=200,
        colorspace=pymupdf.csRGB,
        alpha=False
    )

    image = Image.frombytes(
        "RGB",
        (pix.width, pix.height),
        pix.samples
    )

    extracted_text = pytesseract.image_to_string(image)

    print("OCR Extracted Text:")
    print(extracted_text)

OCR Extracted Text:
Company Leave Policy

Employees receive 20 daysof annual leave,

Leave requests require manager approval




## 7. Clean the Extracted Text

Extracted text may contain unnecessary spaces, tabs, or extra line breaks.

The `clean_text()` function:
- Replaces repeated spaces and tabs with a single space.
- Removes spaces at the beginning and end of each line.
- Reduces three or more consecutive line breaks to two.
- Removes leading and trailing whitespace.

This is basic text normalization. It does not correct OCR spelling mistakes.


In [92]:
def clean_text(text):
    # Replace multiple spaces or tabs with a single space
    text = re.sub(r"[ \t]+", " ", text)

    # Remove spaces at the beginning and end of each line
    text = "\n".join(
        line.strip() for line in text.splitlines()
    )

    # Replace three or more newlines with two newlines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


cleaned_text = clean_text(extracted_text)

print("Cleaned OCR text:")
print(cleaned_text)

Cleaned OCR text:
Company Leave Policy

Employees receive 20 daysof annual leave,

Leave requests require manager approval



## 8. Build an Automatic PDF Extraction Function

This function combines direct PDF text extraction and OCR.

For each page:
1. Try to extract selectable text.
2. If text exists, use direct extraction.
3. If no text exists, render the page as an image and use OCR.
4. Clean the extracted text.
5. Store the page number, extraction method, and text.

The result is a list of dictionaries, one for each PDF page.


In [93]:
def extract_pdf_text(pdf_path):
    results = []

    with pymupdf.open(pdf_path) as document:
        for page_number, page in enumerate(document, start=1):
            # Try direct text extraction
            text = page.get_text().strip()

            if text:
                method = "Direct extraction"
            else:
                # Render page as an image for OCR
                pix = page.get_pixmap(
                    dpi=200,
                    colorspace=pymupdf.csRGB,
                    alpha=False
                )

                image = Image.frombytes(
                    "RGB",
                    (pix.width, pix.height),
                    pix.samples
                )

                text = pytesseract.image_to_string(image)
                method = "OCR"

            text = clean_text(text)

            results.append({
                "page_number": page_number,
                "method": method,
                "text": text
            })

    return results


print("Automatic PDF extraction function created.")

Automatic PDF extraction function created.



## 9. Test Automatic PDF Extraction

Test the function with both a text-based PDF and a scanned PDF.

The expected result is:
- `sample.pdf` uses `Direct extraction`.
- `scanned_sample.pdf` uses `OCR`.

The extraction method is recorded for each page so that the process can be inspected.


In [96]:
test_pdfs = [
    "Documents/sample.pdf",
    "Documents/scanned_sample.pdf"
]

for pdf_path in test_pdfs:
    print(f"\nTesting: {pdf_path}")

    results = extract_pdf_text(pdf_path)

    for page in results:
        print("Page:", page["page_number"])
        print("Method:", page["method"])
        print("Text:")
        print(page["text"])


Testing: Documents/sample.pdf
Page: 1
Method: Direct extraction
Text:
Company Leave Policy - Sample PDF
Document ID: HR-POL-001
Version: 1.0
Annual Leave
Employees receive 20 days of annual leave per calendar year. Leave requests must be submitted
to the employee's manager for approval.
Sick Leave
Employees may take sick leave when they are unwell. They should notify their manager as soon as
reasonably possible.
Public Holidays
Public holidays are handled according to the company's applicable holiday calendar.
This is a fictional sample document created for practicing PDF text extraction. It is not a real company policy.

Testing: Documents/scanned_sample.pdf
Page: 1
Method: OCR
Text:
Company Leave Policy

Employees receive 20 daysof annual leave,

Leave requests require manager approval



## 10. Extract Page-Level Metadata

Metadata describes the source and properties of extracted content.

For each PDF page, record:
- Source filename
- Page number
- Extraction method
- Extracted text

This structured representation helps identify where the text originated.


In [97]:
pdf_path = "Documents/scanned_sample.pdf"
results = extract_pdf_text(pdf_path)

documents = []

for page in results:
    documents.append({
        "source": os.path.basename(pdf_path),
        "page_number": page["page_number"],
        "extraction_method": page["method"],
        "text": page["text"]
    })

print(documents[0])

{'source': 'scanned_sample.pdf', 'page_number': 1, 'extraction_method': 'OCR', 'text': 'Company Leave Policy\n\nEmployees receive 20 daysof annual leave,\n\nLeave requests require manager approval'}



## 11. Validate PDF Files

Before extracting content, check whether a PDF:
- Exists.
- Has the `.pdf` extension.
- Is not an empty file.
- Can be opened by PyMuPDF.
- Contains at least one page.

These checks provide basic file validation. They do not guarantee that the PDF's content is accurate or that every page contains readable text.


In [98]:
def validate_pdf(file_path):
    if not os.path.exists(file_path):
        return False, "File does not exist."

    if not file_path.lower().endswith(".pdf"):
        return False, "File extension must be .pdf."

    if os.path.getsize(file_path) == 0:
        return False, "File is empty."

    try:
        with pymupdf.open(file_path) as document:
            if len(document) == 0:
                return False, "PDF contains no pages."

    except Exception:
        return False, "File is not a valid or readable PDF."

    return True, "Valid PDF."


is_valid, message = validate_pdf("Documents/sample.pdf")

print("Valid:", is_valid)
print("Message:", message)

Valid: True
Message: Valid PDF.



## 12. Extract Text from DOCX Files

The `python-docx` library reads Microsoft Word documents.

The following experiment creates a sample Word document containing headings and paragraphs, then extracts its non-empty paragraphs.

Note: This basic example extracts paragraph text. It does not extract text from tables, images, or other embedded objects.


In [99]:
doc = Document()

doc.add_heading("Company Leave Policy", level=1)

doc.add_heading("Annual Leave", level=2)
doc.add_paragraph(
    "Employees receive 20 days of annual leave per calendar year."
)

doc.add_heading("Sick Leave", level=2)
doc.add_paragraph(
    "Employees should notify their manager when they are unwell."
)

doc.save("Documents/sample.docx")

print("Sample Word document created.")

Sample Word document created.


In [100]:
doc = Document("Documents/sample.docx")

for paragraph in doc.paragraphs:
    if paragraph.text.strip():
        print(paragraph.text)

Company Leave Policy
Annual Leave
Employees receive 20 days of annual leave per calendar year.
Sick Leave
Employees should notify their manager when they are unwell.



## 13. Extract Text from TXT Files

Plain text files can be read directly using Python's built-in file handling.

The file is opened using UTF-8 encoding to support common text characters.


In [101]:
txt_path = "Documents/sample.txt"

with open(txt_path, "w", encoding="utf-8") as file:
    file.write(
        "Company Leave Policy\n"
        "Employees receive 20 days of annual leave.\n"
        "Leave requests require manager approval."
    )

print("Sample TXT file created.")

Sample TXT file created.


In [102]:
with open("Documents/sample.txt", "r", encoding="utf-8") as file:
    text = file.read()

print(text)

Company Leave Policy
Employees receive 20 days of annual leave.
Leave requests require manager approval.



## 14. Create a Unified Document Extraction Function

Different document types require different extraction methods.

The unified function supports:
- PDF: Uses automatic direct extraction and OCR fallback.
- DOCX: Extracts non-empty paragraphs.
- TXT: Reads the file contents.

The function checks the supported extension and file existence before processing the document.


In [103]:
def extract_document_text(file_path):
    extension = os.path.splitext(file_path)[1].lower()

    # Check supported file types
    if extension not in [".pdf", ".docx", ".txt"]:
        raise ValueError(
            f"Unsupported document type: {extension}"
        )

    # Check whether the file exists
    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f"File not found: {file_path}"
        )

    if extension == ".pdf":
        try:
            pages = extract_pdf_text(file_path)
            return "\n\n".join(
                page["text"] for page in pages
            )
        except Exception as error:
            raise ValueError(
                f"Could not process PDF '{file_path}': {error}"
            ) from error

    elif extension == ".docx":
        doc = Document(file_path)
        return "\n".join(
            paragraph.text
            for paragraph in doc.paragraphs
            if paragraph.text.strip()
        )

    elif extension == ".txt":
        with open(file_path, "r", encoding="utf-8") as file:
            return file.read()


print("Unified document extraction function created.")

Unified document extraction function created.



## 15. Test Unified Document Extraction

Run the unified function against PDF, scanned PDF, DOCX, and TXT files.

All four supported formats should return extracted text. The scanned PDF should use OCR internally.


In [104]:
test_files = [
    "Documents/sample.pdf",
    "Documents/scanned_sample.pdf",
    "Documents/sample.docx",
    "Documents/sample.txt"
]

for file_path in test_files:
    print(f"\nTesting: {file_path}")

    text = extract_document_text(file_path)

    print("Extracted characters:", len(text))
    print("Text preview:")
    print(text[:250])


Testing: Documents/sample.pdf
Extracted characters: 558
Text preview:
Company Leave Policy - Sample PDF
Document ID: HR-POL-001
Version: 1.0
Annual Leave
Employees receive 20 days of annual leave per calendar year. Leave requests must be submitted
to the employee's manager for approval.
Sick Leave
Employees may take si

Testing: Documents/scanned_sample.pdf
Extracted characters: 104
Text preview:
Company Leave Policy

Employees receive 20 daysof annual leave,

Leave requests require manager approval

Testing: Documents/sample.docx
Extracted characters: 165
Text preview:
Company Leave Policy
Annual Leave
Employees receive 20 days of annual leave per calendar year.
Sick Leave
Employees should notify their manager when they are unwell.

Testing: Documents/sample.txt
Extracted characters: 104
Text preview:
Company Leave Policy
Employees receive 20 days of annual leave.
Leave requests require manager approval.



## 16. Test Error Handling

Test the extraction function with:
- A missing file.
- An unsupported file extension.
- An invalid PDF file.

The function should raise a clear exception when a document cannot be processed.


In [105]:
# Create a file that is not a real PDF
with open("Documents/invalid.pdf", "w", encoding="utf-8") as file:
    file.write("This is not a real PDF file.")

error_test_files = [
    "Documents/missing.pdf",
    "Documents/sample.xyz",
    "Documents/invalid.pdf"
]

for file_path in error_test_files:
    print(f"\nTesting: {file_path}")

    try:
        text = extract_document_text(file_path)
        print("Extraction succeeded.")

    except Exception as error:
        print(type(error).__name__, "-", error)


Testing: Documents/missing.pdf
FileNotFoundError - File not found: Documents/missing.pdf

Testing: Documents/sample.xyz
ValueError - Unsupported document type: .xyz

Testing: Documents/invalid.pdf
ValueError - Could not process PDF 'Documents/invalid.pdf': Failed to open file 'Documents/invalid.pdf' as type pdf.



## 17. Test an Empty PDF

A PDF may be structurally valid but contain no extractable text.

Create a valid, single-page blank PDF and test the extraction function. The expected extracted text is an empty string.


In [106]:
empty_pdf_path = "Documents/blank.pdf"

with pymupdf.open() as document:
    document.new_page()
    document.save(empty_pdf_path)

print("Blank PDF created.")

text = extract_document_text(empty_pdf_path)

print("Extracted text:", repr(text))

Blank PDF created.
Extracted text: ''



## 18. Build the Metadata-Aware Ingestion Function

The ingestion function combines the previous experiments into one pipeline.

It:
1. Checks the file extension.
2. Checks whether the file exists.
3. Validates PDF files before extraction.
4. Extracts text from the document.
5. Cleans the extracted text.
6. Records metadata.
7. Returns an `empty` status when no text is found.
8. Returns a `success` status when text is extracted.
9. Raises clear errors for unsupported or invalid files.

The output is a Python dictionary containing the status, source, extracted text, metadata, and a message.


In [107]:
def ingest_document(file_path):
    source = os.path.basename(file_path)
    extension = os.path.splitext(file_path)[1].lower()

    # 1. Validate the supported file type
    if extension not in [".pdf", ".docx", ".txt"]:
        raise ValueError(
            f"Unsupported document type: {extension}"
        )

    # 2. Check file existence
    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f"File not found: {file_path}"
        )

    # 3. Extract content and create metadata
    if extension == ".pdf":
        is_valid, message = validate_pdf(file_path)

        if not is_valid:
            raise ValueError(f"Invalid PDF: {message}")

        # Extract PDF text once, retaining page metadata
        pages = extract_pdf_text(file_path)

        text = "\n\n".join(
            page["text"] for page in pages
        )

        metadata = {
            "source": source,
            "file_type": extension,
            "pages": [
                {
                    "page_number": page["page_number"],
                    "extraction_method": page["method"]
                }
                for page in pages
            ]
        }

    else:
        text = extract_document_text(file_path)

        metadata = {
            "source": source,
            "file_type": extension
        }

    # 4. Clean extracted text
    text = clean_text(text)

    # 5. Return an empty result if no text was found
    if not text:
        return {
            "status": "empty",
            "source": source,
            "text": "",
            "metadata": metadata,
            "message": "No extractable text found."
        }

    # 6. Return the successful ingestion result
    return {
        "status": "success",
        "source": source,
        "text": text,
        "metadata": metadata,
        "message": "Document ingested successfully."
    }


print("Metadata-aware ingestion function created.")

Metadata-aware ingestion function created.



## 19. Test Metadata-Aware Ingestion

First, test a normal PDF and inspect the complete result.

Expected metadata includes the filename, file type, page number, and extraction method. The extracted text should be available in the `text` field.


In [108]:
result = ingest_document("Documents/sample.pdf")

print("Status:", result["status"])
print("Source:", result["source"])
print("Metadata:", result["metadata"])
print("Message:", result["message"])
print("Extracted text:")
print(result["text"])

Status: success
Source: sample.pdf
Metadata: {'source': 'sample.pdf', 'file_type': '.pdf', 'pages': [{'page_number': 1, 'extraction_method': 'Direct extraction'}]}
Message: Document ingested successfully.
Extracted text:
Company Leave Policy - Sample PDF
Document ID: HR-POL-001
Version: 1.0
Annual Leave
Employees receive 20 days of annual leave per calendar year. Leave requests must be submitted
to the employee's manager for approval.
Sick Leave
Employees may take sick leave when they are unwell. They should notify their manager as soon as
reasonably possible.
Public Holidays
Public holidays are handled according to the company's applicable holiday calendar.
This is a fictional sample document created for practicing PDF text extraction. It is not a real company policy.



## 20. Test the Final Ingestion Pipeline

Run the ingestion function against all supported document types and edge cases.

Expected behavior:
- Text-based PDF: `success`, using direct extraction.
- Scanned PDF: `success`, using OCR.
- DOCX and TXT: `success`.
- Blank PDF: `empty`.
- Invalid PDF: `ValueError`.
- Missing file: `FileNotFoundError`.
- Unsupported extension: `ValueError`.

Review the actual output before marking the experiment complete.


In [112]:

test_files = [
    "Documents/sample.pdf",
    "Documents/scanned_sample.pdf",
    "Documents/sample.docx",
    "Documents/sample.txt",
    "Documents/blank.pdf",
    "Documents/invalid.pdf",
    "Documents/missing.pdf",
    "Documents/sample.xyz"
]

for file_path in test_files:
    print(f"\n{'=' * 50}")
    print("Testing:", file_path)

    try:
        result = ingest_document(file_path)

        print("Status:", result["status"])
        print("Source:", result["source"])
        print("Metadata:", result["metadata"])
        print("Message:", result["message"])

    except Exception as error:
        print("Error:", type(error).__name__, "-", error)


Testing: Documents/sample.pdf
Status: success
Source: sample.pdf
Metadata: {'source': 'sample.pdf', 'file_type': '.pdf', 'pages': [{'page_number': 1, 'extraction_method': 'Direct extraction'}]}
Message: Document ingested successfully.

Testing: Documents/scanned_sample.pdf
Status: success
Source: scanned_sample.pdf
Metadata: {'source': 'scanned_sample.pdf', 'file_type': '.pdf', 'pages': [{'page_number': 1, 'extraction_method': 'OCR'}]}
Message: Document ingested successfully.

Testing: Documents/sample.docx
Status: success
Source: sample.docx
Metadata: {'source': 'sample.docx', 'file_type': '.docx'}
Message: Document ingested successfully.

Testing: Documents/sample.txt
Status: success
Source: sample.txt
Metadata: {'source': 'sample.txt', 'file_type': '.txt'}
Message: Document ingested successfully.

Testing: Documents/blank.pdf
Status: empty
Source: blank.pdf
Metadata: {'source': 'blank.pdf', 'file_type': '.pdf', 'pages': [{'page_number': 1, 'extraction_method': 'OCR'}]}
Message: No 